In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import r2_score, log_loss

from sklearn.neighbors import KNeighborsRegressor

import os
os.chdir(r"D:\Machine_Learning_CDAC-AI\a] Datasets")

In [5]:
housing = pd.read_csv('Housing.csv')
housing

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [10]:
# Assuming 'price' and 'lotsize' are your targets
X = housing.drop(['price', 'area'], axis=1)
y = housing[['price', 'area']] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [11]:
from sklearn.compose import make_column_selector

ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")


In [12]:
# Your existing OHE logic
X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)


In [13]:
Ks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
scores = []

for k in Ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_trn_ohe, y_train)
    y_pred = knn.predict(X_tst_ohe)
    
    # multioutput='uniform_average' is the default
    avg_r2 = r2_score(y_test, y_pred) 
    scores.append([k, avg_r2])

df_scores = pd.DataFrame(scores, columns=['Ks', 'Avg R2 Score'])
df_scores.sort_values('Avg R2 Score', ascending=False)

,Ks,Avg R2 Score
9,10,0.358782
8,9,0.352831
7,8,0.333959
6,7,0.331587
5,6,0.321619
4,5,0.318001
2,3,0.251097
3,4,0.231761
1,2,0.219679
0,1,0.040255


In [14]:
Ks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
results = []

for k in Ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_trn_ohe, y_train)
    y_pred = knn.predict(X_tst_ohe)
    
    # Get individual scores for [price, lotsize]
    individual_r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    # Get the average score
    avg_r2 = r2_score(y_test, y_pred) 
    
    results.append([k, avg_r2, individual_r2[0], individual_r2[1]])

# Creating a clean DataFrame to view results
df_res = pd.DataFrame(results, columns=['K', 'Avg_R2', 'Price_R2', 'Lotsize_R2'])
df_res.sort_values('Avg_R2', ascending=False)


,K,Avg_R2,Price_R2,Lotsize_R2
9,10,0.358782,0.566290,0.151274
8,9,0.352831,0.546501,0.159161
7,8,0.333959,0.535670,0.132249
6,7,0.331587,0.534731,0.128442
5,6,0.321619,0.540783,0.102455
4,5,0.318001,0.544096,0.091906
2,3,0.251097,0.469909,0.032285
3,4,0.231761,0.460474,0.003047
1,2,0.219679,0.412089,0.027270
0,1,0.040255,0.286817,-0.206308
